In [15]:
import json
import pandas as pd
import os
print(os.getcwd())
# print(os.path.abspath(base_path))


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

/Users/manilesrun/Desktop/Queryanalyser/QA/results_mad_main


In [16]:
# ls

In [17]:
base_path = "../results_mad_main/mmlu_90p"

folders = {

    3: "test_eval_service_3",
    5: "test_eval_service_5",
    8: "test_eval_service_8",
    10: "test_eval_service_10",
    12: "test_eval_service_12",
    14: "test_eval_service_14"

}

In [18]:
data = {}

for k, folder_name in folders.items():

    folder_path = os.path.join(base_path, folder_name)
#     print(folder_path)

    data[k] = []

    for file_name in os.listdir(folder_path):

        if not file_name.endswith(".json"):
            continue

        if "_efficiency" in file_name:
            continue

        file_path = os.path.join(folder_path, file_name)
#         print(file_path)

        with open(file_path, "r") as f:
            rows = json.load(f)
#             print(rows)

        for row in rows:
            row = row.copy()
            row["source_file"] = file_name
            data[k].append(row)

    print(f"K={k}: loaded {len(data[k])} rows")

K=3: loaded 1200 rows
K=5: loaded 1800 rows
K=8: loaded 2700 rows
K=10: loaded 3300 rows
K=12: loaded 3900 rows
K=14: loaded 4500 rows


In [19]:
df_k3 = pd.DataFrame(data[3])
df_k5 = pd.DataFrame(data[5])
df_k8 = pd.DataFrame(data[8])
df_k10 = pd.DataFrame(data[10])
df_k12 = pd.DataFrame(data[12])
df_k14 = pd.DataFrame(data[14])


print("K=3:", df_k3.shape)
print("K=5:", df_k5.shape)
print("K=8:", df_k8.shape)
print("K=10:", df_k10.shape)
print("K=12:", df_k12.shape)
print("K=14:", df_k14.shape)

K=3: (1200, 39)
K=5: (1800, 39)
K=8: (2700, 39)
K=10: (3300, 39)
K=12: (3900, 39)
K=14: (4500, 39)


In [20]:
# df_k3["model_name"].unique()
# df_k3.columns

## combined df

In [21]:
df_combined = pd.concat([df_k3, df_k5, df_k8, df_k10, df_k12, df_k14], ignore_index=True)

print(df_combined.shape)
print(df_combined["total_service_count"].value_counts())

(17400, 39)
14    4500
12    3900
10    3300
8     2700
5     1800
3     1200
Name: total_service_count, dtype: int64


In [22]:
df = df_combined.copy()

df["overall_acc"] = (df["expected_service"] == df["predicted_label"]).astype(float)
df["specialist_recall"] = df["overall_acc"].where(df["expected_service"] != "general")
df["general_recall"]    = df["overall_acc"].where(df["expected_service"] == "general")

out = df.pivot_table(
    index=["routing_method", "model_name", "total_service_count"],
    values=["mad_tau", "mad_cutoff", "specialist_recall", "general_recall", "overall_acc"],
    aggfunc="mean",
)
out["balanced_acc"] = (out["specialist_recall"] + out["general_recall"]) / 2

out = out[["mad_tau", "mad_cutoff", "overall_acc", "specialist_recall", "general_recall", "balanced_acc"]]


pct_cols = ["specialist_recall", "general_recall", "overall_acc", "balanced_acc"]
out[pct_cols] = out[pct_cols] * 100

out.round(1)
# out.isna().sum()

mad_tau  \
routing_method model_name                          total_service_count            
bi-encoder     BAAI/bge-large-en-v1.5              3                        0.0   
                                                   5                        0.0   
                                                   8                        0.0   
                                                   10                       0.0   
                                                   12                       0.0   
                                                   14                       0.0   
cross-encoder  Qwen/Qwen3-Reranker-4B              3                        1.0   
                                                   5                        0.9   
                                                   8                        0.9   
                                                   10                       0.9   
                                                   12                       0.9   
                                                   14                       0.9   
               mixedbread-ai/mxbai-rerank-large-v2 3                        0.0   
                                                   5                        0.0   
                                                   8                        0.0   
                                                   10                       0.0   
                                                   12                       0.0   
                                                   14                       0.0   

                                                                        mad_cutoff  \
routing_method model_name                          total_service_count               
bi-encoder     BAAI/bge-large-en-v1.5              3                           1.6   
                                                   5                           2.0   
                                                   8                           2.4   
                                                   10                          2.6   
                                                   12                          2.7   
                                                   14                          2.9   
cross-encoder  Qwen/Qwen3-Reranker-4B              3                           0.9   
                                                   5                           1.4   
                                                   8                           1.9   
                                                   10                          2.1   
                                                   12                          2.3   
                                                   14                          2.5   
               mixedbread-ai/mxbai-rerank-large-v2 3                           9.7   
                                                   5                          18.9   
                                                   8                          27.3   
                                                   10                         31.3   
                                                   12                         34.6   
                                                   14                         37.4   

                                                                        overall_acc  \
routing_method model_name                          total_service_count                
bi-encoder     BAAI/bge-large-en-v1.5              3                           90.5   
                                                   5                           90.0   
                                                   8                           82.3   
                                                   10                          79.5   
                                                   12                          72.1   
                                                   14                          68

### inspect a model

In [48]:
MODEL = "BAAI/bge-large-en-v1.5"

In [49]:
d3 = df_k3[df_k3["model_name"] == MODEL].copy()

d5 = df_k5[df_k5["model_name"] == MODEL].copy()

d8 = df_k8[df_k8["model_name"] == MODEL].copy()

d10 = df_k10[df_k10["model_name"] == MODEL].copy()
d12 = df_k12[df_k12["model_name"] == MODEL].copy()
d14 = df_k14[df_k14["model_name"] == MODEL].copy()


print("K=3", d3.shape)
print("K=5", d5.shape)
print("K=8", d8.shape)

print("K=10", d10.shape)
print("K=12", d12.shape)
print("K=14", d14.shape)


K=3 (400, 39)
K=5 (600, 39)
K=8 (900, 39)
K=10 (1100, 39)
K=12 (1300, 39)
K=14 (1500, 39)


In [50]:
def expand_scores_and_zscores(df):
    """
    Expand the nested `scores` and `zscores` dict columns
    into normal dataframe columns.
    """

    scores_df = (
        pd.json_normalize(df["scores"])
        .add_suffix("_score")
    )

    zscores_df = (
        pd.json_normalize(df["zscores"])
        .add_suffix("_zscore")
    )

    df_expanded = pd.concat(
        [
            df.reset_index(drop=True),
            scores_df.reset_index(drop=True),
            zscores_df.reset_index(drop=True),
        ],
        axis=1,
    )

    return df_expanded

In [51]:
d3 = expand_scores_and_zscores(d3)
d5 = expand_scores_and_zscores(d5)
d8 = expand_scores_and_zscores(d8)
d10 = expand_scores_and_zscores(d10)
d12 = expand_scores_and_zscores(d12)
d14 = expand_scores_and_zscores(d14)

In [52]:
# d3.columns.to_list()

In [53]:
def recall_summary(df):

    specialist_df = df[
        df["expected_service"] != "general"
    ]

    general_df = df[
        df["expected_service"] == "general"
    ]

    accuracy = df["is_correct"].mean()

    specialist_recall = (
        specialist_df["predicted_label"]
        == specialist_df["expected_service"]
    ).mean()

    general_recall = (
        general_df["predicted_label"]
        == "general"
    ).mean()

    print(
        f"Accuracy: {accuracy:.4f}"
    )

    print(
        f"Specialist Recall: "
        f"{specialist_recall:.4f}"
    )

    print(
        f"General Recall: "
        f"{general_recall:.4f}"
    )


In [54]:
recall_summary(d3)
recall_summary(d5)
recall_summary(d8)
recall_summary(d10)
recall_summary(d12)
recall_summary(d14)


Accuracy: 0.9050
Specialist Recall: 0.9167
General Recall: 0.8700
Accuracy: 0.9000
Specialist Recall: 0.9020
General Recall: 0.8900
Accuracy: 0.8233
Specialist Recall: 0.8137
General Recall: 0.9000
Accuracy: 0.7945
Specialist Recall: 0.7810
General Recall: 0.9300
Accuracy: 0.7208
Specialist Recall: 0.7142
General Recall: 0.8000
Accuracy: 0.6853
Specialist Recall: 0.6771
General Recall: 0.8000


In [55]:
recall_summary(d3)

Accuracy: 0.9050
Specialist Recall: 0.9167
General Recall: 0.8700


In [56]:
cm_k3 = pd.crosstab(
    d3["expected_service"],
    d3["predicted_label"]
)

cm_k3

predicted_label,abstract_algebra,anatomy,astronomy,general
expected_service,,,,
abstract_algebra,100,0,0,0
anatomy,0,79,0,21
astronomy,1,0,96,3
general,12,0,1,87


In [59]:
error_k3 = d3[ (d3["expected_service"] == "general") & (d3["predicted_label"] == "abstract_algebra")]

In [60]:
error_k3[
    [
        "query",
        "expected_service",
        "predicted_label",

#         "abstract_algebra_zscore",
#         "anatomy_zscore",
#         "astronomy_zscore",

#         "top_zscore",
#         "second_top_zscore",
#         "zscore_margin",

#         "std_score",

        "abstract_algebra_score",
        "anatomy_score",
        "astronomy_score",
    ]
]

,query,expected_service,predicted_label,abstract_algebra_score,anatomy_score,astronomy_score
302,what is my card's apr rate,general,abstract_algebra,0.375085,0.261893,0.336628
305,Where are cards delivered to?,general,abstract_algebra,0.376542,0.312142,0.296283
306,Can I change my information?,general,abstract_algebra,0.365122,0.276881,0.309708
308,Can I add money with Apple Pay?,general,abstract_algebra,0.318832,0.232156,0.244091
311,"If I feel someone has my card information, can I get a new card?",general,abstract_algebra,0.318907,0.240769,0.257824
315,Why did I have to pay extra because I paid with card?,general,abstract_algebra,0.339233,0.247010,0.288934
322,What type of ATMs accept this card?,general,abstract_algebra,0.340776,0.258323,0.283335
323,The balance on my account wasn't updated after I made a depost.,general,abstract_algebra,0.351609,0.250895,0.314534
333,"Can you help me with a weird charge? It's a pound charge that never goes away from the statement view on the app I'm using. It's not described as anything but ""Pending"", and that status has never changed during the last two days.",general,abstract_algebra,0.419431,0.323504,0.366770
335,How can I fix a problem where contactless isn't working?,general,abstract_algebra,0.330076,0.242827,0.275585


# K5

In [34]:
recall_summary(d5)

Accuracy: 0.9600
Specialist Recall: 0.9700
General Recall: 0.9100


In [35]:
cm_k5 = pd.crosstab(
    d5["expected_service"],
    d5["predicted_label"]
)
cm_k5

predicted_label,abstract_algebra,anatomy,astronomy,business_ethics,college_computer_science,general
expected_service,,,,,,
abstract_algebra,99,0,0,0,0,1
anatomy,0,100,0,0,0,0
astronomy,1,1,97,0,0,1
business_ethics,1,0,0,95,1,3
college_computer_science,5,0,0,0,94,1
general,0,2,3,1,3,91


In [36]:
# error_k5 = d5[ (d5["expected_service"] == "general") & (d5["predicted_label"] == "anatomy")]

In [37]:
# error_k5[
#     [
#         "query",
#         "expected_service",
#         "predicted_label",

#         "abstract_algebra_zscore",
#         "anatomy_zscore",
#         "astronomy_zscore",
#         "business_ethics_zscore",
#         "college_computer_science_zscore",

# #         "top_zscore",
# #         "second_top_zscore",
#         "zscore_margin",

#         "std_score",

#         "abstract_algebra_score",
#         "anatomy_score",
#         "astronomy_score",
#         "business_ethics_score",
#         "college_computer_science_score",
#     ]
# ]

# K8

In [38]:
recall_summary(d8)

Accuracy: 0.8878
Specialist Recall: 0.8950
General Recall: 0.8300


In [39]:
cm_k8 = pd.crosstab(
    d8["expected_service"],
    d8["predicted_label"]
)

cm_k8

predicted_label,abstract_algebra,anatomy,astronomy,business_ethics,college_computer_science,general,high_school_chemistry,high_school_geography,jurisprudence
expected_service,,,,,,,,,
abstract_algebra,99,0,0,0,0,1,0,0,0
anatomy,0,98,0,0,0,0,2,0,0
astronomy,0,0,97,0,0,3,0,0,0
business_ethics,0,0,0,85,1,5,0,1,8
college_computer_science,6,0,0,0,93,1,0,0,0
general,0,2,3,1,4,83,0,0,7
high_school_chemistry,2,2,1,0,0,0,95,0,0
high_school_geography,1,1,4,3,0,34,0,52,5
jurisprudence,0,0,0,0,0,3,0,0,97


In [40]:
# error_k8 = d8[ (d8["expected_service"] == "general") & (d8["predicted_label"] == "anatomy")]

In [41]:
# error_k8[
#     [
#         "query",
#         "expected_service",
#         "predicted_label",

#         "abstract_algebra_zscore",
#         "anatomy_zscore",
#         "astronomy_zscore",
#         "business_ethics_zscore",
#         "college_computer_science_zscore",
#         "high_school_chemistry_zscore",
#         "high_school_geography_zscore",
#         "jurisprudence_zscore",

# #         "top_zscore",
# #         "second_top_zscore",
#         "zscore_margin",

#         "std_score",

#         "abstract_algebra_score",
#         "anatomy_score",
#         "astronomy_score",
#         "business_ethics_score",
#         "college_computer_science_score",
#         "high_school_chemistry_score",
#         "high_school_geography_score",
#         "jurisprudence_score",
#     ]
# ]

# K10

In [42]:
recall_summary(d10)

Accuracy: 0.8782
Specialist Recall: 0.8790
General Recall: 0.8700


In [43]:
cm_k10 = pd.crosstab(
    d10["expected_service"],
    d10["predicted_label"]
)

cm_k10

predicted_label,abstract_algebra,anatomy,astronomy,business_ethics,college_computer_science,formal_logic,general,high_school_chemistry,high_school_geography,jurisprudence,world_religions
expected_service,,,,,,,,,,,
abstract_algebra,94,0,0,0,0,6,0,0,0,0,0
anatomy,0,99,0,0,0,0,0,1,0,0,0
astronomy,0,0,97,0,0,1,2,0,0,0,0
business_ethics,0,0,0,83,1,1,4,0,0,7,4
college_computer_science,5,0,0,0,90,4,1,0,0,0,0
formal_logic,0,0,0,0,0,96,1,0,0,1,2
general,0,1,2,1,2,0,87,0,0,6,1
high_school_chemistry,0,2,0,0,0,0,0,98,0,0,0
high_school_geography,1,0,2,4,0,1,32,0,45,4,11


In [46]:
error_k10 = d10[ (d10["expected_service"] == "high_school_geography") & (d10["predicted_label"] == "general")]

In [47]:
error_k10[
    [
        "query",
        "expected_service",
        "predicted_label",

#         "abstract_algebra_zscore",
#         "anatomy_zscore",
#         "astronomy_zscore",
#         "business_ethics_zscore",
#         "college_computer_science_zscore",
#         "formal_logic_zscore",
#         "high_school_chemistry_zscore",
#         "high_school_geography_zscore",
#         "jurisprudence_zscore",
#         "world_religions_zscore",

#         "top_zscore",
#         "second_top_zscore",
#         "zscore_margin",

#         "std_score",

        "abstract_algebra_score",
        "anatomy_score",
        "astronomy_score",
        "business_ethics_score",
        "college_computer_science_score",
        "formal_logic_score",
        "high_school_chemistry_score",
        "high_school_geography_score",
        "jurisprudence_score",
        "world_religions_score",
    ]
]

,query,expected_service,predicted_label,abstract_algebra_score,anatomy_score,astronomy_score,business_ethics_score,college_computer_science_score,formal_logic_score,high_school_chemistry_score,high_school_geography_score,jurisprudence_score,world_religions_score
701,"Olives, grapes, and chickpeas are staple crops of which type of farming?",high_school_geography,general,-13.6875,-12.1250,-12.6875,-13.7500,-13.3750,-14.3125,-12.7500,-11.6250,-13.5625,-12.8125
702,The majority of Kurds are found in which country?,high_school_geography,general,-14.0625,-13.1875,-13.6250,-14.1875,-13.8750,-14.3125,-13.7500,-12.1250,-14.0000,-12.6875
707,A map that presents a single class of statistics is called a,high_school_geography,general,-10.2500,-11.2500,-10.0625,-12.0625,-10.6250,-11.0000,-11.8750,-10.3750,-11.5000,-12.3125
715,A chief factor that contributed to the Second Agricultural Revolution was,high_school_geography,general,-11.6250,-12.6875,-11.8750,-13.1250,-12.8125,-13.4375,-12.9375,-12.6875,-12.2500,-11.9375
720,Which of the following statements is NOT part of Gunnar Myrdal's theory of cumulative causation?,high_school_geography,general,-11.6875,-12.9375,-11.8125,-12.5625,-11.3750,-11.5000,-12.6250,-11.6875,-10.0000,-11.9375
721,"Part of the boundary between the United States and Mexico is the Rio Grande, an example of",high_school_geography,general,-12.3125,-11.0000,-10.4375,-13.4375,-12.7500,-13.3750,-12.0625,-11.0000,-10.2500,-11.3125
726,The purpose for the creation of NATO was to,high_school_geography,general,-12.6875,-12.7500,-12.2500,-13.3750,-12.6250,-13.2500,-13.4375,-13.3750,-12.5000,-12.1875
729,Which factor in the last quarter century greatly destabilized Eastern Europe and led to ethnic conflict?,high_school_geography,general,-12.0000,-12.1875,-12.3750,-12.4375,-13.0625,-14.0000,-12.9375,-12.0625,-11.8750,-11.5625
730,Opponents of genetically modified organisms (GMOs) in food are afraid that the GMOs,high_school_geography,general,-13.0625,-11.2500,-11.8750,-12.0000,-12.8750,-12.5000,-12.0625,-13.3750,-11.4375,-12.1250
731,What family do most European languages belong to?,high_school_geography,general,-13.0000,-12.6250,-12.8750,-14.1250,-13.1875,-12.5625,-13.3125,-11.6875,-13.1250,-11.9375
